In [140]:
import os
import glob
import re
import json
import time
import datetime
import xcdat as xc
import numpy as np
import pandas as pd
import multiprocessing
import logging

import collections
from collections import OrderedDict

import psutil
import subprocess
from itertools import chain
from subprocess import Popen, PIPE, call

import pcmdi_metrics

In [141]:
def load_metrics(metric_file='synthetic_metrics_list.json'):
    """Load metrics from the JSON file."""
    try:
        with open(metric_file) as f:
            return json.load(f)
    except FileNotFoundError as e:
        logging.error(f"File not found: {e}")
        raise
    except json.JSONDecodeError as e:
        logging.error(f"Error decoding JSON: {e}")
        raise

In [ ]:
def prioritize_files(file_list, prefix='ts_'):
    """Ensure that files starting with 'ts_' come first in the list."""
    ts_files = [f for f in file_list if os.path.basename(f).startswith(prefix)]
    other_files = [f for f in file_list if f not in ts_files]
    return ts_files + other_files

In [142]:
def update_cmip_metrics_old(parameter, version, fixed_vars):
    if fixed_vars is None:
        return

    cmip_name_old = parameter['cmip_name'].split(".")
    cmip_name_new = parameter['cmip_new'].split(".")
    cmip_path = parameter['cmip_path']
    output_path = parameter.get('output_path', cmip_path)
    
    for var in fixed_vars:
        # Find old and new files
        cmip_fold = glob.glob(os.path.join(
            cmip_path,
            cmip_name_old[0], cmip_name_old[1], cmip_name_old[2],
            f"{var}*.{cmip_name_old[2]}.json"
        ))
        cmip_fnew = glob.glob(os.path.join(
            cmip_path,
            cmip_name_new[0], cmip_name_new[1], cmip_name_new[2],
            f"{var}*_{cmip_name_new[2]}.json"
        ))
        
        if not cmip_fold or not cmip_fnew:
            print(f"[WARN] Missing file for variable {var}")
            continue

        old_file = cmip_fold[0]
        new_file = cmip_fnew[0]
        
        cmip_out = old_file.replace(cmip_name_old[2],version)
        print(f'base file {old_file}')
        print(f'revise file {new_file}')
        print(f'output file {cmip_out}')
        
        with open(old_file, 'r') as f:
            df_old = json.load(f)
        with open(new_file, 'r') as f:
            df_new = json.load(f)

        # Update values in df_old with matching entries from df_new
        for model in df_old['RESULTS']:
            if model not in df_new['RESULTS']:
                continue
            for obskey in df_old['RESULTS'][model]:
                if obskey not in df_new['RESULTS'][model]:
                    continue
                for obssrc in df_old['RESULTS'][model][obskey]:
                    if obssrc == 'source':
                        continue 
                    if obssrc not in df_new['RESULTS'][model][obskey]:
                        continue
                    for reg in df_old['RESULTS'][model][obskey][obssrc]:
                        if reg == 'InputClimatologyFileName':
                            continue 
                        if reg not in df_new['RESULTS'][model][obskey][obssrc]:
                            continue
                        print('approach to: ',model,obskey,obssrc,reg)
                        # Replace value
                        new_val = df_new['RESULTS'][model][obskey][obssrc][reg]
                        df_old['RESULTS'][model][obskey][obssrc][reg] = new_val
                        print(f"[INFO] Updated: {model}/{obskey}/{obssrc}/{reg}")

        # Save updated output
        out_file = os.path.join(output_path, f"{cmip_out}")
        with open(out_file, 'w') as fout:
            json.dump(df_old, fout, indent=2)
        print(f"[INFO] Saved merged metrics for {var} to {out_file}")


In [145]:
def compare_json_files(old_file, new_file):
    with open(old_file, 'r') as f:
        old_data = json.load(f)
    with open(new_file, 'r') as f:
        new_data = json.load(f)

    print(f"[INFO] Comparing:\n  OLD: {old_file}\n  NEW: {new_file}")

    def recurse_compare(d1, d2, path=""):
        if isinstance(d1, dict) and isinstance(d2, dict):
            all_keys = set(d1.keys()).union(d2.keys())
            for key in sorted(all_keys):
                new_path = f"{path}/{key}" if path else key
                if key not in d1:
                    print(f"[ADDED]   {new_path}: {d2[key]}")
                elif key not in d2:
                    print(f"[REMOVED] {new_path}: {d1[key]}")
                else:
                    recurse_compare(d1[key], d2[key], new_path)
        else:
            if d1 != d2:
                print(f"[CHANGED] {path}:\n   OLD: {d1}\n   NEW: {d2}")

    recurse_compare(old_data, new_data)

In [148]:
def update_cmip_metrics(parameter, version, fixed_vars):
    if fixed_vars is None:
        return

    cmip_name_old = parameter['cmip_name'].split(".")
    cmip_name_new = parameter['cmip_new'].split(".")
    cmip_path = parameter['cmip_path']
    output_path = os.path.join(
            cmip_path, cmip_name_old[0], cmip_name_old[1], version
    )
    
    parameter.get('output_path', cmip_path)

    for var in fixed_vars:
        # Locate old and new files
        cmip_fold = glob.glob(os.path.join(
            cmip_path, cmip_name_old[0], cmip_name_old[1], cmip_name_old[2],
            f"{var}*.{cmip_name_old[2]}.json"
        ))
        cmip_fnew = glob.glob(os.path.join(
            cmip_path, cmip_name_new[0], cmip_name_new[1], cmip_name_new[2],
            f"{var}*_{cmip_name_new[2]}.json"
        ))

        if not cmip_fold or not cmip_fnew:
            print(f"[WARN] Missing file for variable '{var}'")
            continue

        old_file = cmip_fold[0]
        new_file = cmip_fnew[0]
        
        out_file_name = os.path.basename(old_file).replace(cmip_name_old[2], version)
        out_file = os.path.join(output_path, out_file_name)
        
        compare_json_files(old_file,out_file)
        print(xxxx)
        
        print(f"[INFO] Base file:   {old_file}")
        print(f"[INFO] Update from: {new_file}")
        print(f"[INFO] Output to:   {out_file}")

        with open(old_file, 'r') as f:
            df_old = json.load(f)
        with open(new_file, 'r') as f:
            df_new = json.load(f)

        # === Update metrics ===
        for model in df_old.get('RESULTS', {}):
            if model not in df_new.get('RESULTS', {}):
                continue
            for obskey in df_old['RESULTS'][model]:
                if obskey not in df_new['RESULTS'][model]:
                    continue
                for obssrc in df_old['RESULTS'][model][obskey]:
                    if obssrc == 'source':
                        continue
                    if obssrc not in df_new['RESULTS'][model][obskey]:
                        continue
                    for reg in df_old['RESULTS'][model][obskey][obssrc]:
                        if reg == 'InputClimatologyFileName':
                            continue
                        if reg not in df_new['RESULTS'][model][obskey][obssrc]:
                            continue

                        # Replace value
                        new_val = df_new['RESULTS'][model][obskey][obssrc][reg]
                        df_old['RESULTS'][model][obskey][obssrc][reg] = new_val
                        print(f"[INFO] Updated: {model}/{obskey}/{obssrc}/{reg}")

        # === Save merged output ===
        os.makedirs(output_path, exist_ok=True)
        if os.path.exists(out_file):
            os.remove(out_file)
            
        with open(out_file, 'w') as fout:
            json.dump(df_old, fout, indent=2)
        print(f"[SUCCESS] Merged file saved: {out_file}")


In [149]:
if __name__ == "__main__":
    """Main function to drive the entire process."""
    # Load metrics and setup parameters
    parameter = configure_parameters()
    
    # Process metrics
    """Process metrics and generate plots."""
    test_input_path = os.path.join(
        '/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/e3sm-pcmdi',
        'v3.LR.amip_*',
        'pcmdi_diags',
        'model_vs_obs',
        'metrics_data', '%(group_type)'
    )
    parameter['cmip_path'] = '/lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/mean_climate'
    parameter['cmip_name'] = 'cmip6.amip.v20241029' #v20250702
    parameter['cmip_new'] = 'cmip6.amip.v20250630' #v20250702
    
    update_cmip_metrics(parameter,version='v20250702',fixed_vars=['tauu','tauv']) 
    

[INFO] Comparing:
  OLD: /lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/mean_climate/cmip6/amip/v20241029/tauu.cmip6.amip.regrid2.2p5x2p5.v20241029.json
  NEW: /lcrc/group/e3sm/diagnostics/pcmdi_data/metrics_data/mean_climate/cmip6/amip/v20250702/tauu.cmip6.amip.regrid2.2p5x2p5.v20250702.json
[CHANGED] RESULTS/CAS-ESM2-0/default/r1i1p1f1/NHEX/bias_xy/CalendarMonths:
   OLD: ['-9.77616e-02', '-8.81120e-02', '-7.52070e-02', '-5.35146e-02', '-3.30621e-02', '-2.70557e-02', '-2.23182e-02', '-2.32859e-02', '-3.86098e-02', '-6.27322e-02', '-8.25705e-02', '-9.50296e-02']
   NEW: ['-4.57680e-02', '-4.13918e-02', '-3.69619e-02', '-2.89364e-02', '-2.36730e-02', '-2.16000e-02', '-1.95030e-02', '-2.12470e-02', '-2.80457e-02', '-3.27235e-02', '-3.88616e-02', '-4.57567e-02']
[CHANGED] RESULTS/CAS-ESM2-0/default/r1i1p1f1/NHEX/bias_xy/ann:
   OLD: -5.82716e-02
   NEW: -3.20391e-02
[CHANGED] RESULTS/CAS-ESM2-0/default/r1i1p1f1/NHEX/bias_xy/djf:
   OLD: -9.36192e-02
   NEW: -4.42897e-02
[CHANGED] R

NameError: name 'xxxx' is not defined